# Обучение и проверка в scikit-learn

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Обучать модель, получать предсказания и сравнивать качество с baseline.

## Единый интерфейс

В [первом эксперименте KNN](../../../ml/nearest-neighbors/selection-evaluation/) мы сами считали расстояния, голоса и долю верных ответов. Теперь сохраним этот протокол, а вычисления поручим библиотеке. Здесь разбираем интерфейс; подробный выбор метрик и схем проверки будет в [главе об оценке данных](../../../ml/evaluation-preprocessing/).

scikit-learn — библиотека готовых моделей и средств их проверки. Признаки описывают объект, ответ — то, что требуется предсказать. В классификации ответ выбирается из категорий — классов. Baseline (базовое решение) — простой способ, с которым сравнивают более сложную модель.

В scikit-learn `fit(X, y)` обучает по признакам и ответам, `predict(X)` выдаёт предсказания. Преобразователь учится через `fit`, а применяет выученное правило через `transform`. Например, масштабирование запоминает среднее и меру разброса каждого обучающего признака, затем использует их без пересчёта. Метод вызывается у объекта через точку: `model.fit(X, y)` обучает конкретную модель с именем `model`.

Таблица `X` имеет форму «объекты × признаки», `y` — один ответ на объект. Исключите из признаков правильный ответ, идентификатор и служебный признак разбиения. Хороший результат на обучении не заменяет проверку на новых объектах.

## Разбиение и метрика

Отделим тест — часть, которую не используем для выбора модели. Во время разработки подбираем параметры на обучающей части с внутренней **перекрёстной проверкой**: делим эту часть на несколько групп, по очереди оставляем одну для проверки, а на остальных обучаемся. Сравниваем средние оценки по всем таким проверкам. Тест в них не участвует. После выбора один раз измеряем качество на тесте. Для классификации доля верных ответов `accuracy` — понятная начальная метрика, но при редком классе она может вводить в заблуждение.

Пример ниже создаёт новый набор из 24 вымышленных объектов, а не читает `gliders.csv` из первого эксперимента. В этой отдельной задаче метки 0 и 1 заданы другим учебным правилом; результаты двух наборов нельзя сравнивать как оценки одной модели. Размах не меньше 22 см задаёт класс 1, меньший — класс 0; масса и бумага нужны для демонстрации подготовки. Это простая задача для проверки кода, а не реалистичная оценка модели полёта.

`train_test_split` делит признаки и ответы согласованно, возвращая обучающие и тестовые части. `test_size=0.25` выделяет четверть объектов в тест. `stratify=y` сохраняет доли классов настолько близко, насколько позволяют целые числа объектов, `random_state=17` фиксирует начальное состояние случайного разбиения. Случайное разделение подходит не всем задачам: повторные измерения одного объекта надо держать вместе, временные данные проверять на будущем. Иначе тест может оказаться слишком лёгким.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

X = pd.DataFrame({
    "span_cm": list(range(10, 34)),
    "mass_g": [8., 9., 10., 11., None, 12.] * 4,
    "paper": ["тонкая", "плотная"] * 12,
})
y = np.array([0] * 12 + [1] * 12)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=17,
)
assert len(X_train) == 18 and len(X_test) == 6


## Разобранный пример: модель и постоянный ответ

`KNeighborsClassifier(n_neighbors=3)` создаёт объект модели с тремя соседями: это вызов конструктора, который пока ничего не обучает. KNN (метод ближайших соседей) находит три ближайших обучающих объекта и выбирает класс большинством голосов. В одном признаке расстояние — модуль разности размахов. Например, у соседей с метками `[0, 1, 1]` победит 1. `DummyClassifier(strategy="most_frequent")` запоминает наиболее частый класс и отвечает им на каждый объект. `accuracy_score` вычисляет долю верных ответов. Для первого опыта возьмём только столбец размаха: в нём нет пропусков и не требуется подготовка разных типов признаков.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train[["span_cm"]], y_train)
predicted = model.predict(X_test[["span_cm"]])
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
assert predicted.shape == y_test.shape
assert accuracy_score(y_test, baseline.predict(X_test)) == 0.5
print("Доля верных ответов:", accuracy_score(y_test, predicted))


Двойные квадратные скобки выбирают таблицу с одним столбцом; одиночные дали бы Series. Для моделей важно сохранить двумерную форму X и одномерную y. Шесть тестовых объектов дают грубую оценку: одна ошибка меняет долю на одну шестую.

## Как пользоваться тестом

После просмотра теста изменение числа соседей становится подбором по тестовым ответам. Для разработки используйте отдельную проверочную часть или перекрёстную проверку внутри train. Метрика сама по себе не защищает от одинаковых объектов в обеих частях и признаков, вычисленных по правильному ответу.

## Задачи

### 1. Базовая

Что делает fit?

<details>
<summary>Подсказка</summary>

Это шаг обучения.

</details>



### 2. Базовая

Что делает predict?

<details>
<summary>Подсказка</summary>

Обучение уже завершено.

</details>



### 3. Прикладная

Проверьте число тестовых предсказаний и допустимые метки.

<details>
<summary>Подсказка</summary>

Используйте длину и множество.

</details>



### 4. Прикладная

Для ответов [0, 1, 1, 0] и предсказаний [0, 0, 1, 0] вычислите accuracy.

<details>
<summary>Подсказка</summary>

Посчитайте совпадения в одинаковых позициях и разделите на общее число ответов.

</details>



### 5. Олимпиадная

Модель всегда выдаёт 0 при 99% нулей. Почему accuracy 0.99 недостаточно?

<details>
<summary>Подсказка</summary>

Проверьте редкий класс и baseline.

</details>



### 6. Олимпиадная

Почему нельзя много раз выбирать число соседей по одному тесту и считать его независимой проверкой?

<details>
<summary>Подсказка</summary>

Тестовые ответы участвовали в выборе.

</details>



## Итоги

Вы умеете создать модель, обучить её и сравнить предсказания с постоянным базовым решением.

## Самопроверка

Объясните разницу между созданием объекта модели и вызовом `fit`. Почему идентификатор и правильный ответ исключают из признаков? Что меняется в accuracy при одной ошибке на шести объектах?

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

Определяет состояние модели по обучающим данным.

### Задача 2

Применяет модель к новым объектам и возвращает предсказания.

### Задача 3

In [ ]:
assert len(predicted) == len(X_test)
assert set(predicted).issubset({0, 1})


### Задача 4

In [ ]:
assert accuracy_score([0, 1, 1, 0], [0, 0, 1, 0]) == 0.75


### Задача 5

Модель не находит ни одного объекта класса 1 и совпадает с простейшим постоянным ответом. Нужны ошибки по классам и цель задачи.

### Задача 6

Решение подстраивается под особенности теста. Нужна другая итоговая проверка и отдельные данные для подбора.